# FINANCIAL HEALTH INDEX (FHI) PREDICTION
---
* **Competition : Predict SME Financial Health Index for Southern African businesses**
* **Target      : Low / Medium / High (multiclass classification)**
* **Metric      : Macro F1 Score**
* **Best LB**     : `0.8919`

**Pipeline Overview:**

1. Data cleaning & preprocessing
2. Feature engineering
3. Target encoding for country
4. SMOTE oversampling for minority class
5. XGBoost + LightGBM + CatBoost + MLP ensemble
6. Pseudo-labeling on high-confidence test predictions
7. Threshold optimization


In [52]:
# =============================================================================
# REPRODUCIBILITY NOTE
# =============================================================================
# Results in this notebook are hardware-dependent for XGBoost:
#
#   GPU execution : LB score = 0.8919  (best result)
#   CPU execution : LB score = 0.8876
#
# LightGBM, CatBoost, and MLP are unaffected by hardware.
# The difference arises from floating point parallelization in XGBoost's
# hist tree method — GPU and CPU use different numerical precision handling
# which produces different split points and therefore different predictions.
#
# To reproduce the best result (0.8919), run on a machine with a GPU and
# set tree_method="hist" (default). For CPU-only machines, results will
# differ slightly but the pipeline remains identical.
# =============================================================================

## Imports & Configuration

In [53]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# =============================================================================
# CONFIGURATION
# =============================================================================
SEED       = 42
N_FOLDS    = 10          # 10-fold CV for stable OOF estimates
TARGET_COL = "Target"
ID_COL     = "ID"

# Label order — LabelEncoder sorts alphabetically so:
# High=0, Low=1, Medium=2
LABEL_ORDER = ["Low", "Medium", "High"]

DATA_DIR   = Path(".")           # directory containing Train.csv and Test.csv
OUTPUT_DIR = Path("./outputs")   # directory for saving OOF files and submissions
OUTPUT_DIR.mkdir(exist_ok=True)

le_target = LabelEncoder()
le_target.fit(LABEL_ORDER)

print("=" * 60)
print("FINANCIAL HEALTH INDEX PREDICTION")
print("=" * 60)
print(f"Target classes : {le_target.classes_}")
print(f"               : High=0, Low=1, Medium=2 (alphabetical)")
print(f"N_FOLDS        : {N_FOLDS}")
print(f"Output dir     : {OUTPUT_DIR}")

FINANCIAL HEALTH INDEX PREDICTION
Target classes : ['High' 'Low' 'Medium']
               : High=0, Low=1, Medium=2 (alphabetical)
N_FOLDS        : 10
Output dir     : outputs


## Load Data

In [54]:
# =============================================================================
# LOAD DATA
# =============================================================================
train = pd.read_csv(DATA_DIR / "Train.csv")
test  = pd.read_csv(DATA_DIR / "Test.csv")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nTarget distribution:")
print(train[TARGET_COL].value_counts())
print(f"\nClass imbalance ratio — Low:Medium:High = "
      f"{train[TARGET_COL].value_counts()['Low']}:"
      f"{train[TARGET_COL].value_counts()['Medium']}:"
      f"{train[TARGET_COL].value_counts()['High']}")
print(f"\nCountry distribution:")
print(train["country"].value_counts())

Train shape : (9618, 39)
Test shape  : (2405, 38)

Target distribution:
Target
Low       6280
Medium    2868
High       470
Name: count, dtype: int64

Class imbalance ratio — Low:Medium:High = 6280:2868:470

Country distribution:
country
eswatini    2674
zimbabwe    2612
malawi      2388
lesotho     1944
Name: count, dtype: int64


## Data Cleaning & Preprocessing

In [55]:
# =============================================================================
# DATA CLEANING & PREPROCESSING
#
# Key issues found during data audit:
#   1. Apostrophe variants — curly (') vs straight (') apostrophes cause
#      silent NaN mapping in status columns across 9 columns
#   2. Mixed values — current_problem_cash_flow contains "0" alongside
#      "Yes"/"No" from data entry inconsistencies
#   3. Status columns — "Have now", "Never had", "Used to have" have
#      natural ordinal meaning and are encoded accordingly
#   4. High missingness — 20 columns have 20-47% missing values;
#      the pattern of who didn't answer is itself informative
# =============================================================================

def preprocess(df):
    """
    Clean and encode raw survey data.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe with Target and ID columns already removed

    Returns
    -------
    pd.DataFrame
        Cleaned and encoded dataframe ready for feature engineering
    """
    df = df.copy()

    # -----------------------------------------------------------------------
    # Fix 1: Normalize apostrophe variants and strip whitespace
    # Curly apostrophe (Unicode 8217) appears alongside straight apostrophe
    # (Unicode 39) in status columns — causes silent NaN mapping
    # -----------------------------------------------------------------------
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.replace('\u2019', "'", regex=False).str.strip()

    # -----------------------------------------------------------------------
    # Fix 2: Status columns → ordinal encoding with _ord suffix
    # Natural ordering: Have now (2) > Used to have (1) > Never had (0)
    # "Don't know" responses encoded as -1 to distinguish from missing (NaN)
    # -----------------------------------------------------------------------
    status_map = {
        "Have now"                       : 2,
        "Used to have but don't have now": 1,
        "Never had"                      : 0,
        "Don't know"                     : -1,
        ""                               : np.nan
    }
    status_cols = [
        "motor_vehicle_insurance", "has_mobile_money", "has_credit_card",
        "has_loan_account", "has_internet_banking", "has_debit_card",
        "medical_insurance", "funeral_insurance",
        "uses_friends_family_savings", "uses_informal_lender"
    ]
    for col in status_cols:
        if col in df.columns:
            df[f"{col}_ord"] = df[col].map(status_map)
            df.drop(columns=[col], inplace=True)

    # -----------------------------------------------------------------------
    # Fix 3: Binary Yes/No columns → 1/0/-1
    # "Don't know" variants unified to -1 to preserve the response signal
    # "0" in current_problem_cash_flow treated as "No" (data entry artifact)
    # -----------------------------------------------------------------------
    binary_map = {
        "Yes"                       :  1,
        "No"                        :  0,
        "Don't know or N/A"         : -1,
        "Don't know"                : -1,
        "Don?t know / doesn?t apply": -1,
        "Refused"                   : -1,
        "0"                         :  0,
        ""                          : np.nan
    }
    binary_cols = [
        "attitude_stable_business_environment", "attitude_worried_shutdown",
        "compliance_income_tax", "perception_insurance_doesnt_cover_losses",
        "perception_cannot_afford_insurance", "has_cellphone", "owner_sex",
        "attitude_satisfied_with_achievement", "keeps_financial_records",
        "perception_insurance_companies_dont_insure_businesses_like_yours",
        "perception_insurance_important", "has_insurance", "covid_essential_service",
        "attitude_more_successful_next_year", "problem_sourcing_money",
        "marketing_word_of_mouth", "future_risk_theft_stock",
        "motivation_make_more_money", "current_problem_cash_flow"
    ]
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].map(binary_map)

    # -----------------------------------------------------------------------
    # Fix 4: offers_credit_to_customers → ordinal 0/1/2
    # -----------------------------------------------------------------------
    credit_map = {
        "Yes, always"   : 2, "Yes, always " : 2, "Yes, Always"   : 2,
        "Yes, sometimes": 1, "Yes, sometimes": 1, "Yes, Sometimes": 1,
        "No"            : 0, ""             : np.nan
    }
    if "offers_credit_to_customers" in df.columns:
        df["offers_credit_to_customers"] = df["offers_credit_to_customers"].map(
            lambda x: credit_map.get(x, np.nan) if not pd.isna(x) else np.nan
        )

    # -----------------------------------------------------------------------
    # Fix 5: owner_sex → binary (Male=1, Female=0)
    # -----------------------------------------------------------------------
    if "owner_sex" in df.columns:
        df["owner_sex"] = df["owner_sex"].map({"Male": 1, "Female": 0, "": np.nan})

    # -----------------------------------------------------------------------
    # Country → one-hot encoding
    # Will be replaced by target encoding in the next step
    # -----------------------------------------------------------------------
    df = pd.get_dummies(df, columns=["country"], prefix="country", drop_first=False)

    return df


print("Preprocessing function defined.")
print("Fixes applied:")
print("  [1] Apostrophe normalization (curly → straight)")
print("  [2] Status columns → ordinal with _ord suffix")
print("  [3] Binary Yes/No → 1/0/-1 (unified don't know variants)")
print("  [4] offers_credit_to_customers → ordinal 0/1/2")
print("  [5] owner_sex → binary 1/0")
print("  [6] Country → one-hot (target encoded in Cell 4)")

Preprocessing function defined.
Fixes applied:
  [1] Apostrophe normalization (curly → straight)
  [2] Status columns → ordinal with _ord suffix
  [3] Binary Yes/No → 1/0/-1 (unified don't know variants)
  [4] offers_credit_to_customers → ordinal 0/1/2
  [5] owner_sex → binary 1/0
  [6] Country → one-hot (target encoded in Cell 4)


## Feature Engineering

In [56]:
# =============================================================================
# FEATURE ENGINEERING
#
# Features added on top of preprocessed data:
#   - Log-transformed financials: handles extreme currency skew
#     (personal_income ranges from 0 to 150M with median of 2K)
#   - Financial ratios: profit proxy, expense ratio, income efficiency
#   - Business age in total months: combines years + months fields
#   - Missing value flags: missingness pattern is informative
#     (missing_business_age_years was top SHAP feature)
#   - Financial product count: more products = higher FHI
#   - Attitude score: optimistic owners tend to have higher FHI
# =============================================================================

def engineer_features(df):
    """
    Add engineered features on top of preprocessed data.

    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed dataframe from preprocess()

    Returns
    -------
    pd.DataFrame
        Dataframe with additional engineered features
    """
    df = df.copy()

    # -----------------------------------------------------------------------
    # Log-transform skewed financial columns
    # Multi-currency data (MWK, SZL, ZWL, LSL) creates extreme outliers
    # Log1p transform compresses the scale while preserving ordering
    # -----------------------------------------------------------------------
    for col in ["personal_income", "business_expenses", "business_turnover"]:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(df[col].fillna(0))

    # -----------------------------------------------------------------------
    # Financial ratio features
    # These ratios capture relative financial health independent of currency
    # -----------------------------------------------------------------------
    eps = 1e-6  # small constant to avoid division by zero

    # Profit proxy: how much revenue exceeds expenses
    df["profit_proxy"]     = (df["business_turnover"].fillna(0) -
                               df["business_expenses"].fillna(0))
    df["log_profit_proxy"] = np.log1p(np.maximum(df["profit_proxy"], 0))

    # Expense efficiency: what fraction of revenue goes to expenses
    df["expense_ratio"]    = (df["business_expenses"].fillna(0) /
                               (df["business_turnover"].fillna(0) + eps))

    # Income relative to business scale
    df["income_to_turnover"] = (df["personal_income"].fillna(0) /
                                 (df["business_turnover"].fillna(0) + eps))
    df["income_to_expenses"] = (df["personal_income"].fillna(0) /
                                 (df["business_expenses"].fillna(0) + eps))

    # -----------------------------------------------------------------------
    # Business age in total months
    # Combines business_age_years and business_age_months into one feature
    # -----------------------------------------------------------------------
    df["total_business_months"] = (
        df["business_age_years"].fillna(0) * 12 +
        df["business_age_months"].fillna(0)
    )

    # -----------------------------------------------------------------------
    # Missing value flags for high-missingness columns
    # Many columns have 20-47% missing — WHO didn't answer is informative
    # e.g. Lesotho respondents have systematic missingness on many columns
    # -----------------------------------------------------------------------
    high_missing_cols = [
        "motor_vehicle_insurance_ord", "has_mobile_money_ord",
        "current_problem_cash_flow",   "has_cellphone",
        "has_loan_account_ord",        "has_internet_banking_ord",
        "has_debit_card_ord",          "future_risk_theft_stock",
        "medical_insurance_ord",       "funeral_insurance_ord",
        "motivation_make_more_money",  "uses_friends_family_savings_ord",
        "uses_informal_lender_ord",    "business_age_months",
        "business_age_years"
    ]
    for col in high_missing_cols:
        if col in df.columns:
            df[f"missing_{col}"] = df[col].isna().astype(int)

    # -----------------------------------------------------------------------
    # Count of financial products currently owned
    # Businesses with more active products (value=2) tend to have higher FHI
    # -----------------------------------------------------------------------
    product_cols = [
        "motor_vehicle_insurance_ord", "has_mobile_money_ord",
        "has_credit_card_ord",         "has_loan_account_ord",
        "has_internet_banking_ord",    "has_debit_card_ord",
        "medical_insurance_ord",       "funeral_insurance_ord"
    ]
    present_products = [c for c in product_cols if c in df.columns]
    df["n_financial_products"] = df[present_products].apply(
        lambda row: (row == 2).sum(), axis=1
    )

    # -----------------------------------------------------------------------
    # Positive attitude score
    # Sum of positive attitudes — optimistic owners tend to have higher FHI
    # -----------------------------------------------------------------------
    attitude_cols = [
        "attitude_stable_business_environment",
        "attitude_more_successful_next_year",
        "attitude_satisfied_with_achievement"
    ]
    present_attitudes = [c for c in attitude_cols if c in df.columns]
    df["attitude_score"] = df[present_attitudes].apply(
        lambda row: (row == 1).sum(), axis=1
    )

# -----------------------------------------------------------------------
    # Business vs Personal income ratio
    # Captures whether owner relies on business or business is the powerhouse
    # Adding 1 to denominator avoids division by zero
    # Log-transformed to handle extreme values when personal_income ~ 0
    # -----------------------------------------------------------------------
    if all(c in df.columns for c in ["business_turnover", "personal_income"]):
        df["business_vs_personal_ratio"] = (
            df["business_turnover"].fillna(0) /
            (df["personal_income"].fillna(0) + 1)
        )
        df["log_business_vs_personal_ratio"] = np.log1p(
            np.maximum(df["business_vs_personal_ratio"], 0)
        )

    # -----------------------------------------------------------------------
    # Fintech adoption score
    # Sum of digital financial tool usage — higher score = more digitally
    # integrated business, tends to correlate with higher FHI
    # Note: columns renamed to _ord suffix during preprocessing
    # Values: 2=Have now, 1=Used to have, 0=Never had, -1=Don't know
    # We count only active users (value == 2)
    # -----------------------------------------------------------------------
    fintech_cols = [
        "has_mobile_money_ord", "has_internet_banking_ord",
        "has_debit_card_ord",   "has_credit_card_ord"
    ]
    present_fintech = [c for c in fintech_cols if c in df.columns]
    if present_fintech:
        df["fintech_adoption_score"] = df[present_fintech].apply(
            lambda row: (row == 2).sum(), axis=1
        )    

# -----------------------------------------------------------------------
    # Insurance sophistication score
    # Count of active insurance products — businesses with more insurance
    # coverage tend to be more financially resilient (higher FHI)
    # Only counts currently active (value == 2)
    # -----------------------------------------------------------------------
    insurance_cols = [
        "motor_vehicle_insurance_ord",
        "medical_insurance_ord",
        "funeral_insurance_ord"
    ]
    present_insurance = [c for c in insurance_cols if c in df.columns]
    if present_insurance:
        df["insurance_sophistication_score"] = df[present_insurance].apply(
            lambda row: (row == 2).sum(), axis=1
        )

    # -----------------------------------------------------------------------
    # Formal credit access score
    # Count of formal credit/banking products currently active
    # Higher score = more integrated into formal financial system
    # -----------------------------------------------------------------------
    formal_credit_cols = [
        "has_loan_account_ord",
        "has_credit_card_ord",
        "has_internet_banking_ord"
    ]
    present_formal = [c for c in formal_credit_cols if c in df.columns]
    if present_formal:
        df["formal_credit_score"] = df[present_formal].apply(
            lambda row: (row == 2).sum(), axis=1
        )

    # -----------------------------------------------------------------------
    # Informal finance reliance score
    # Count of informal financial products currently used
    # Higher score = more reliant on informal finance = inverse FHI signal
    # Businesses using informal lenders and family savings tend to have
    # lower access to formal financial services
    # -----------------------------------------------------------------------
    informal_cols = [
        "uses_friends_family_savings_ord",
        "uses_informal_lender_ord"
    ]
    present_informal = [c for c in informal_cols if c in df.columns]
    if present_informal:
        df["informal_finance_reliance"] = df[present_informal].apply(
            lambda row: (row == 2).sum(), axis=1
        )

    # -----------------------------------------------------------------------
    # Formal vs informal finance ratio
    # Captures the balance between formal and informal financial access
    # High ratio = more formal = higher FHI expected
    # -----------------------------------------------------------------------
    if "formal_credit_score" in df.columns and "informal_finance_reliance" in df.columns:
        df["formal_vs_informal_ratio"] = (
            df["formal_credit_score"] /
            (df["informal_finance_reliance"] + 1)
        )

    # -----------------------------------------------------------------------
    # Overall financial access score
    # Combines fintech, insurance, and formal credit into one composite score
    # Weighted sum reflecting relative SHAP importance from earlier analysis:
    #   medical_insurance > has_credit_card > has_mobile_money > funeral_insurance
    # -----------------------------------------------------------------------
    if all(c in df.columns for c in ["fintech_adoption_score",
                                      "insurance_sophistication_score",
                                      "formal_credit_score"]):
        df["overall_financial_access"] = (
            df["fintech_adoption_score"]          * 1.0 +
            df["insurance_sophistication_score"]  * 1.2 +  # slightly higher weight
            df["formal_credit_score"]             * 1.1    # based on SHAP importance
        )
    
    return df


# Apply preprocessing + feature engineering
train_fe = engineer_features(preprocess(train.drop(columns=[TARGET_COL, ID_COL])))
test_fe  = engineer_features(preprocess(test.drop(columns=[ID_COL])))

# Align train and test columns — handles any one-hot encoding discrepancies
train_fe, test_fe = train_fe.align(test_fe, join="left", axis=1, fill_value=0)

# Encode target labels
y = le_target.transform(train[TARGET_COL])

# Impute remaining NaNs with median — fit on train only to prevent leakage
imputer = SimpleImputer(strategy="median")
X       = imputer.fit_transform(train_fe.values)
X_test  = imputer.transform(test_fe.values)

feature_names = list(train_fe.columns)

print(f"Features after engineering : {len(feature_names)}")
print(f"NaN remaining in X         : {np.isnan(X).sum()}")
print(f"NaN remaining in X_test    : {np.isnan(X_test).sum()}")
print(f"\nFeature groups:")
print(f"  Original encoded features  : {len([f for f in feature_names if 'missing_' not in f and 'log_' not in f and f not in ['profit_proxy', 'log_profit_proxy', 'expense_ratio', 'income_to_turnover', 'income_to_expenses', 'total_business_months', 'n_financial_products', 'attitude_score']])}")
print(f"  Log-transformed financials : {len([f for f in feature_names if 'log_' in f])}")
print(f"  Financial ratios           : {len([f for f in feature_names if f in ['profit_proxy', 'expense_ratio', 'income_to_turnover', 'income_to_expenses']])}")
print(f"  Missing value flags        : {len([f for f in feature_names if 'missing_' in f])}")

Features after engineering : 74
NaN remaining in X         : 0
NaN remaining in X_test    : 0

Feature groups:
  Original encoded features  : 47
  Log-transformed financials : 5
  Financial ratios           : 4
  Missing value flags        : 15


## Target Encoding for Country

In [57]:
# =============================================================================
# TARGET ENCODING FOR COUNTRY
#
# Why target encoding instead of one-hot:
#   Country is highly predictive of FHI — Eswatini has 11.5% High businesses
#   vs Lesotho's 0.3%. Target encoding captures this directly as P(High|country)
#   rather than treating countries as independent binary flags.
#
# Implementation:
#   - Out-of-fold encoding on train prevents data leakage
#   - Each country column replaced with P(High) for that country
#   - Test encoding uses full training set mean (no leakage risk)
# =============================================================================

def target_encode_country(X, y, X_test, feature_names, n_folds=N_FOLDS, seed=SEED):
    """
    Replace one-hot country columns with out-of-fold P(High) target encoding.

    Parameters
    ----------
    X           : np.ndarray, training features
    y           : np.ndarray, encoded target labels
    X_test      : np.ndarray, test features
    feature_names : list, feature names corresponding to X columns
    n_folds     : int, number of CV folds for OOF encoding
    seed        : int, random seed

    Returns
    -------
    X_enc, X_test_enc : np.ndarray, arrays with country columns target-encoded
    """
    country_col_idx = [i for i, f in enumerate(feature_names)
                       if f.startswith("country_")]
    country_names   = [feature_names[i] for i in country_col_idx]

    X_enc      = X.copy().astype(float)
    X_test_enc = X_test.copy().astype(float)

    skf_enc = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for col_idx in country_col_idx:
        # OOF encoding: for each fold, compute P(High | country) using
        # only training fold rows — validation fold uses held-out estimate
        oof_enc = np.zeros(len(y))

        for tr_idx, val_idx in skf_enc.split(X, y):
            country_in_fold = X[tr_idx, col_idx] == 1
            if country_in_fold.sum() > 0:
                # P(High) = fraction of High class among this country in train fold
                p_high = np.mean(y[tr_idx][country_in_fold] == 0)
            else:
                p_high = np.mean(y == 0)  # global fallback if no samples
            oof_enc[val_idx] = p_high

        X_enc[:, col_idx] = oof_enc

        # Test encoding: use full training set mean (safe — no leakage)
        country_in_train = X[:, col_idx] == 1
        if country_in_train.sum() > 0:
            X_test_enc[:, col_idx] = np.mean(y[country_in_train] == 0)
        else:
            X_test_enc[:, col_idx] = np.mean(y == 0)

    # Print encoded values per country for verification
    print("Country target encoding (P(High) per country):")
    for i, (col_idx, name) in enumerate(zip(country_col_idx, country_names)):
        train_enc = X_enc[X[:, col_idx] == 1, col_idx].mean()
        test_enc  = X_test_enc[X_test[:, col_idx] == 1, col_idx].mean()
        print(f"  {name:<25} train={train_enc:.4f}, test={test_enc:.4f}")

    return X_enc, X_test_enc


X_enc, X_test_enc = target_encode_country(X, y, X_test, feature_names)

print(f"\nX_enc shape      : {X_enc.shape}")
print(f"X_test_enc shape : {X_test_enc.shape}")

Country target encoding (P(High) per country):
  country_eswatini          train=0.0031, test=0.0031
  country_lesotho           train=0.0402, test=0.0402
  country_malawi            train=0.0234, test=0.0234
  country_zimbabwe          train=nan, test=nan

X_enc shape      : (9618, 73)
X_test_enc shape : (2405, 73)


## CV, SMOTE & Utility Functions

In [58]:
# =============================================================================
# CV, SMOTE & UTILITY FUNCTIONS
#
# SMOTE strategy:
#   "minority" — oversample only the smallest class (High, 470 samples)
#   up to the size of the next smallest class (Medium, 2868 samples).
#   Applied inside each fold on training data only — validation always
#   uses original unaugmented data to keep OOF scores honest.
#
# Threshold optimization:
#   Instead of argmax on raw probabilities, we scale each class probability
#   by a learned threshold before taking argmax. This allows the model to
#   be more or less aggressive about predicting each class, optimizing
#   directly for macro F1.
# =============================================================================

# 10-fold stratified CV — more stable OOF than 5-fold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# SMOTE — oversample High class to match Medium class size
sm  = SMOTE(sampling_strategy="minority", random_state=SEED, k_neighbors=5)


def macro_f1(y_true, y_pred):
    """Compute macro-averaged F1 score."""
    return f1_score(y_true, y_pred, average="macro")


def apply_thresholds(probs, thresholds):
    """
    Scale class probabilities by per-class thresholds then take argmax.
    Lower threshold = model more willing to predict that class.
    """
    return np.argmax(probs / np.array(thresholds), axis=1)


def neg_macro_f1(thresholds, probs, y_true):
    """Negative macro F1 for minimization."""
    return -macro_f1(y_true, apply_thresholds(probs, thresholds))


def optimize_thresholds(oof_probs, y):
    """
    Find optimal per-class thresholds to maximize macro F1 on OOF predictions.
    Uses Nelder-Mead simplex optimization with bounds [0.1, 2.0] per class.
    """
    result = minimize(
        neg_macro_f1,
        x0=[1.0, 1.0, 1.0],
        args=(oof_probs, y),
        method="Nelder-Mead",
        bounds=[(0.1, 2.0)] * 3,
        options={"maxiter": 5000, "xatol": 1e-5, "fatol": 1e-5}
    )
    return result.x, -result.fun


def save_oof(name, oof, test_preds):
    """Save OOF and test predictions to disk for fast reloading."""
    np.save(OUTPUT_DIR / f"oof_{name}.npy", oof)
    np.save(OUTPUT_DIR / f"test_{name}.npy", test_preds)
    print(f"  Saved: oof_{name}.npy + test_{name}.npy")


def load_oof(name):
    """Load previously saved OOF and test predictions."""
    oof  = np.load(OUTPUT_DIR / f"oof_{name}.npy")
    test = np.load(OUTPUT_DIR / f"test_{name}.npy")
    return oof, test


def make_submission(test_probs, thresholds, filename):
    """Generate submission file from test probabilities and thresholds."""
    preds = le_target.inverse_transform(apply_thresholds(test_probs, thresholds))
    sub   = pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: preds})
    sub.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"  Saved: {OUTPUT_DIR / filename}")
    print(f"  Prediction distribution:\n{pd.Series(preds).value_counts().to_string()}")


def log_experiment(name, model_scores, ensemble_f1, optimized_f1, lb_f1=None, notes=""):
    """Append experiment results to JSON log file."""
    log_path = OUTPUT_DIR / "experiment_log.json"
    entry = {
        "timestamp"        : datetime.now().strftime("%Y-%m-%d %H:%M"),
        "experiment"       : name,
        "n_folds"          : N_FOLDS,
        "model_oof_scores" : {k: round(v, 4) for k, v in model_scores.items()},
        "ensemble_oof_f1"  : round(ensemble_f1, 4),
        "optimized_oof_f1" : round(optimized_f1, 4),
        "lb_f1"            : lb_f1,
        "notes"            : notes
    }
    try:
        with open(log_path) as f:
            log = json.load(f)
    except FileNotFoundError:
        log = []
    log.append(entry)
    with open(log_path, "w") as f:
        json.dump(log, f, indent=2)
    print(f"  Experiment logged → {log_path}")


def get_orig_val_idx(val_idx, orig_len):
    """Return only validation indices belonging to original training data."""
    return val_idx[val_idx < orig_len]


print("Utilities ready.")
print(f"SMOTE strategy : minority (High → matches Medium count per fold)")
print(f"CV folds       : {N_FOLDS}-fold stratified")

Utilities ready.
SMOTE strategy : minority (High → matches Medium count per fold)
CV folds       : 10-fold stratified


## XGBoost OOF Training

In [59]:
# =============================================================================
# XGBOOST — OUT-OF-FOLD TRAINING
#
# Parameters selected based on competition experiments:
#   - hist tree method: fast and memory efficient for tabular data
#   - Moderate depth (6) and regularization to prevent overfitting
#   - Early stopping on validation log loss to find optimal n_estimators
#   - Balanced sample weights to handle class imbalance alongside SMOTE
# =============================================================================

xgb_params = {
    "objective"        : "multi:softprob",  # returns probabilities
    "num_class"        : 3,
    "eval_metric"      : "mlogloss",
    "verbosity"        : 0,
    "use_label_encoder": False,
    "random_state"     : SEED,
    "tree_method"      : "hist",            # fast histogram-based algorithm
    "device"           : "cuda",             # set to "cpu" if no GPU available
    "n_estimators"     : 800,
    "learning_rate"    : 0.03,              # low LR with more trees
    "max_depth"        : 6,
    "min_child_weight" : 5,
    "subsample"        : 0.8,               # row subsampling
    "colsample_bytree" : 0.7,               # feature subsampling
    "gamma"            : 0.01,              # min split loss
    "reg_alpha"        : 0.1,              # L1 regularization
    "reg_lambda"       : 1.0,              # L2 regularization
}

print("Training XGBoost (10-fold OOF + SMOTE)...")
oof_xgb  = np.zeros((len(y), 3))
test_xgb = np.zeros((len(X_test_enc), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_enc, y)):
    # Apply SMOTE on training fold only — validation uses original data
    X_tr, y_tr = sm.fit_resample(X_enc[tr_idx], y[tr_idx])

    # Compute sample weights after SMOTE (class distribution has changed)
    sw = compute_sample_weight("balanced", y_tr)

    m = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
    m.fit(X_tr, y_tr,
          sample_weight=sw,
          eval_set=[(X_enc[val_idx], y[val_idx])],
          verbose=False)

    oof_xgb[val_idx] = m.predict_proba(X_enc[val_idx])
    test_xgb        += m.predict_proba(X_test_enc) / N_FOLDS

    fold_f1 = macro_f1(y[val_idx], np.argmax(oof_xgb[val_idx], axis=1))
    print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")

xgb_score = macro_f1(y, np.argmax(oof_xgb, axis=1))
print(f"\n[XGB] OOF Macro F1: {xgb_score:.4f}")
print(classification_report(y, np.argmax(oof_xgb, axis=1),
                             target_names=le_target.classes_))
save_oof("xgb", oof_xgb, test_xgb)

Training XGBoost (10-fold OOF + SMOTE)...
  Fold 01 F1: 0.8060
  Fold 02 F1: 0.8107
  Fold 03 F1: 0.8223
  Fold 04 F1: 0.8195
  Fold 05 F1: 0.7830
  Fold 06 F1: 0.7908
  Fold 07 F1: 0.8159
  Fold 08 F1: 0.8030
  Fold 09 F1: 0.8424
  Fold 10 F1: 0.7550

[XGB] OOF Macro F1: 0.8052
              precision    recall  f1-score   support

        High       0.85      0.63      0.72       470
         Low       0.92      0.91      0.91      6280
      Medium       0.76      0.81      0.78      2868

    accuracy                           0.86      9618
   macro avg       0.84      0.78      0.81      9618
weighted avg       0.87      0.86      0.86      9618

  Saved: oof_xgb.npy + test_xgb.npy


## LightGBM OOF Training

In [60]:
# =============================================================================
# LIGHTGBM — OUT-OF-FOLD TRAINING
#
# Parameters selected based on competition experiments:
#   - Leaf-wise tree growth (num_leaves=80) captures complex patterns
#   - class_weight="balanced" provides additional imbalance correction
#   - LGB callbacks used instead of early_stopping_rounds parameter
#     (API difference from XGBoost)
# =============================================================================

lgb_params = {
    "objective"        : "multiclass",
    "num_class"        : 3,
    "metric"           : "multi_logloss",
    "verbosity"        : -1,
    "boosting_type"    : "gbdt",
    "random_state"     : SEED,
    "class_weight"     : "balanced",        # handles imbalance alongside SMOTE
    "n_estimators"     : 800,
    "learning_rate"    : 0.03,
    "num_leaves"       : 80,                # leaf-wise growth
    "max_depth"        : 8,
    "min_child_samples": 20,                # min samples per leaf
    "subsample"        : 0.8,
    "colsample_bytree" : 0.7,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.0,
    "min_split_gain"   : 0.01,
}

print("Training LightGBM (10-fold OOF + SMOTE)...")
oof_lgb  = np.zeros((len(y), 3))
test_lgb = np.zeros((len(X_test_enc), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_enc, y)):
    X_tr, y_tr = sm.fit_resample(X_enc[tr_idx], y[tr_idx])

    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_enc[val_idx], y[val_idx])],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(-1)])

    oof_lgb[val_idx] = m.predict_proba(X_enc[val_idx])
    test_lgb        += m.predict_proba(X_test_enc) / N_FOLDS

    fold_f1 = macro_f1(y[val_idx], np.argmax(oof_lgb[val_idx], axis=1))
    print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")

lgb_score = macro_f1(y, np.argmax(oof_lgb, axis=1))
print(f"\n[LGB] OOF Macro F1: {lgb_score:.4f}")
print(classification_report(y, np.argmax(oof_lgb, axis=1),
                             target_names=le_target.classes_))
save_oof("lgb", oof_lgb, test_lgb)

Training LightGBM (10-fold OOF + SMOTE)...
  Fold 01 F1: 0.7972
  Fold 02 F1: 0.7934
  Fold 03 F1: 0.8349
  Fold 04 F1: 0.8132
  Fold 05 F1: 0.7983
  Fold 06 F1: 0.7777
  Fold 07 F1: 0.8118
  Fold 08 F1: 0.8058
  Fold 09 F1: 0.8300
  Fold 10 F1: 0.7635

[LGB] OOF Macro F1: 0.8029
              precision    recall  f1-score   support

        High       0.85      0.62      0.72       470
         Low       0.92      0.90      0.91      6280
      Medium       0.75      0.81      0.78      2868

    accuracy                           0.86      9618
   macro avg       0.84      0.78      0.80      9618
weighted avg       0.86      0.86      0.86      9618

  Saved: oof_lgb.npy + test_lgb.npy


## CatBoost OOF Training

In [61]:
# =============================================================================
# CATBOOST — OUT-OF-FOLD TRAINING
#
# Parameters selected based on competition experiments:
#   - auto_class_weights="Balanced" handles imbalance alongside SMOTE
#   - border_count=128 provides fine-grained feature splits
#   - bagging_temperature controls randomness in bootstrap sampling
# =============================================================================

cat_params = {
    "loss_function"      : "MultiClass",
    "eval_metric"        : "TotalF1",       # optimize directly for F1
    "random_seed"        : SEED,
    "verbose"            : 0,
    "auto_class_weights" : "Balanced",
    "iterations"         : 800,
    "learning_rate"      : 0.03,
    "depth"              : 6,
    "l2_leaf_reg"        : 1.0,
    "bagging_temperature": 0.5,
    "random_strength"    : 1.0,
    "border_count"       : 128,             # number of splits per feature
}

print("Training CatBoost (10-fold OOF + SMOTE)...")
oof_cat  = np.zeros((len(y), 3))
test_cat = np.zeros((len(X_test_enc), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_enc, y)):
    X_tr, y_tr = sm.fit_resample(X_enc[tr_idx], y[tr_idx])

    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr,
          eval_set=(X_enc[val_idx], y[val_idx]),
          early_stopping_rounds=50)

    oof_cat[val_idx] = m.predict_proba(X_enc[val_idx])
    test_cat        += m.predict_proba(X_test_enc) / N_FOLDS

    fold_f1 = macro_f1(y[val_idx], np.argmax(oof_cat[val_idx], axis=1))
    print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")

cat_score = macro_f1(y, np.argmax(oof_cat, axis=1))
print(f"\n[CAT] OOF Macro F1: {cat_score:.4f}")
print(classification_report(y, np.argmax(oof_cat, axis=1),
                             target_names=le_target.classes_))
save_oof("cat", oof_cat, test_cat)

Training CatBoost (10-fold OOF + SMOTE)...
  Fold 01 F1: 0.7990
  Fold 02 F1: 0.7737
  Fold 03 F1: 0.8100
  Fold 04 F1: 0.7961
  Fold 05 F1: 0.7690
  Fold 06 F1: 0.7576
  Fold 07 F1: 0.8423
  Fold 08 F1: 0.7696
  Fold 09 F1: 0.8363
  Fold 10 F1: 0.7698

[CAT] OOF Macro F1: 0.7925
              precision    recall  f1-score   support

        High       0.75      0.66      0.70       470
         Low       0.93      0.88      0.90      6280
      Medium       0.72      0.84      0.77      2868

    accuracy                           0.85      9618
   macro avg       0.80      0.79      0.79      9618
weighted avg       0.86      0.85      0.86      9618

  Saved: oof_cat.npy + test_cat.npy


## MLP OOF Training

In [62]:
# =============================================================================
# MLP CLASSIFIER — OUT-OF-FOLD TRAINING
#
# Role in ensemble:
#   MLP adds diversity as the only non-tree-based model. Despite being
#   the weakest individual model, it contributes complementary predictions
#   that improve ensemble performance.
#
# Key differences from tree models:
#   - Requires feature scaling (StandardScaler) — trees are scale-invariant
#   - No class_weight parameter — imbalance handled entirely via SMOTE
#   - Early stopping via validation_fraction to prevent overfitting
# =============================================================================

# Scale features to zero mean, unit variance — required for MLP
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X_enc)
X_test_scaled = scaler.transform(X_test_enc)

mlp_params = {
    "hidden_layer_sizes" : (256, 128, 64),  # 3-layer network
    "activation"         : "relu",
    "solver"             : "adam",
    "alpha"              : 0.001,           # L2 regularization weight
    "learning_rate_init" : 0.001,
    "max_iter"           : 500,
    "early_stopping"     : True,            # stop when val loss plateaus
    "validation_fraction": 0.1,
    "n_iter_no_change"   : 20,              # patience
    "random_state"       : SEED,
    # Note: MLPClassifier does not support class_weight parameter
    # Class imbalance handled via SMOTE in the training loop
}

print("Training MLP (10-fold OOF + SMOTE)...")
oof_mlp  = np.zeros((len(y), 3))
test_mlp = np.zeros((len(X_test_scaled), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    # Apply SMOTE on scaled training fold
    X_tr, y_tr = sm.fit_resample(X_scaled[tr_idx], y[tr_idx])

    m = MLPClassifier(**mlp_params)
    m.fit(X_tr, y_tr)

    oof_mlp[val_idx] = m.predict_proba(X_scaled[val_idx])
    test_mlp        += m.predict_proba(X_test_scaled) / N_FOLDS

    fold_f1 = macro_f1(y[val_idx], np.argmax(oof_mlp[val_idx], axis=1))
    print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")

mlp_score = macro_f1(y, np.argmax(oof_mlp, axis=1))
print(f"\n[MLP] OOF Macro F1: {mlp_score:.4f}")
print(classification_report(y, np.argmax(oof_mlp, axis=1),
                             target_names=le_target.classes_))
save_oof("mlp", oof_mlp, test_mlp)

Training MLP (10-fold OOF + SMOTE)...
  Fold 01 F1: 0.7390
  Fold 02 F1: 0.7653
  Fold 03 F1: 0.7734
  Fold 04 F1: 0.8015
  Fold 05 F1: 0.7633
  Fold 06 F1: 0.7731
  Fold 07 F1: 0.7473
  Fold 08 F1: 0.7592
  Fold 09 F1: 0.8083
  Fold 10 F1: 0.7753

[MLP] OOF Macro F1: 0.7711
              precision    recall  f1-score   support

        High       0.66      0.62      0.64       470
         Low       0.88      0.96      0.92      6280
      Medium       0.83      0.68      0.75      2868

    accuracy                           0.86      9618
   macro avg       0.79      0.76      0.77      9618
weighted avg       0.86      0.86      0.86      9618

  Saved: oof_mlp.npy + test_mlp.npy


## Pseudo-labeling

In [63]:
# =============================================================================
# PSEUDO-LABELING
#
# Strategy:
#   Use the initial ensemble to predict labels for test samples. Only
#   high-confidence predictions (max probability >= 0.95) are added to
#   the training set. This leverages unlabeled test data to improve
#   generalization.
#
# Why 0.95 threshold:
#   Experimentation showed 0.95 maximizes LB score. Lower thresholds
#   (0.93) add noisy labels; higher thresholds (0.97, 0.98) add too
#   few samples to be beneficial. Iterative pseudo-labeling (2 rounds)
#   was also tested but hurt performance due to compounding label errors.
#
# Important:
#   OOF scores after pseudo-labeling are computed on original training
#   samples only (indices < orig_len) to keep evaluation honest.
# =============================================================================

PSEUDO_THRESHOLD = 0.95

# Load initial OOF predictions from all four models
oof_xgb, test_xgb = load_oof("xgb")
oof_lgb, test_lgb = load_oof("lgb")
oof_cat, test_cat = load_oof("cat")
oof_mlp, test_mlp = load_oof("mlp")

xgb_score = macro_f1(y, np.argmax(oof_xgb, axis=1))
lgb_score = macro_f1(y, np.argmax(oof_lgb, axis=1))
cat_score = macro_f1(y, np.argmax(oof_cat, axis=1))
mlp_score = macro_f1(y, np.argmax(oof_mlp, axis=1))

print("Initial OOF scores (before pseudo-labeling):")
print(f"  XGBoost   : {xgb_score:.4f}")
print(f"  LightGBM  : {lgb_score:.4f}")
print(f"  CatBoost  : {cat_score:.4f}")
print(f"  MLP       : {mlp_score:.4f}")

# Build ensemble probabilities for pseudo-label generation
# Weights proportional to OOF F1 score
scores_init   = np.array([xgb_score, lgb_score, cat_score, mlp_score])
weights_init  = scores_init / scores_init.sum()
test_probs_pl = (weights_init[0]*test_xgb + weights_init[1]*test_lgb +
                 weights_init[2]*test_cat  + weights_init[3]*test_mlp)

# Select high-confidence predictions only
max_probs      = test_probs_pl.max(axis=1)
confident_mask = max_probs >= PSEUDO_THRESHOLD
pseudo_labels  = np.argmax(test_probs_pl[confident_mask], axis=1)

print(f"\nPseudo-labeling (threshold={PSEUDO_THRESHOLD}):")
print(f"  Total test samples    : {len(confident_mask)}")
print(f"  Confident predictions : {confident_mask.sum()} "
      f"({confident_mask.mean()*100:.1f}%)")
print(f"  Pseudo label distribution:")
print(pd.Series(le_target.inverse_transform(pseudo_labels))
      .value_counts().to_string())

# Build augmented training set
X_enc_pl = np.vstack([X_enc,  X_test_enc[confident_mask]])
y_pl     = np.concatenate([y, pseudo_labels])

# Scale augmented data for MLP
scaler_pl         = StandardScaler()
X_enc_pl_scaled   = scaler_pl.fit_transform(X_enc_pl)
X_test_pl_scaled  = scaler_pl.transform(X_test_enc)

print(f"\nAugmented training set:")
print(f"  Original size : {len(y)}")
print(f"  New size      : {len(y_pl)}")
print(f"  Added samples : {len(y_pl) - len(y)}")

Initial OOF scores (before pseudo-labeling):
  XGBoost   : 0.8052
  LightGBM  : 0.8029
  CatBoost  : 0.7925
  MLP       : 0.7711

Pseudo-labeling (threshold=0.95):
  Total test samples    : 2405
  Confident predictions : 422 (17.5%)
  Pseudo label distribution:
Medium    232
Low       136
High       54

Augmented training set:
  Original size : 9618
  New size      : 10040
  Added samples : 422


##  Retrain on Pseudo-labeled Data

In [64]:
# =============================================================================
# RETRAIN ALL MODELS ON PSEUDO-LABELED DATA
#
# All four models retrained with the expanded dataset.
# OOF scores computed on original training samples only (idx < orig_len)
# to ensure evaluation reflects true generalization performance.
# =============================================================================

orig_len = len(y)
skf_pl   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# -----------------------------------------------------------------------
# XGBoost — retrain with pseudo-labeled data
# -----------------------------------------------------------------------
print("Retraining XGBoost on pseudo-labeled data...")
oof_xgb_pl  = np.zeros((orig_len, 3))
test_xgb_pl = np.zeros((len(X_test_enc), 3))

for fold, (tr_idx, val_idx) in enumerate(skf_pl.split(X_enc_pl, y_pl)):
    val_orig   = get_orig_val_idx(val_idx, orig_len)
    X_tr, y_tr = sm.fit_resample(X_enc_pl[tr_idx], y_pl[tr_idx])
    sw = compute_sample_weight("balanced", y_tr)

    m = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
    m.fit(X_tr, y_tr,
          sample_weight=sw,
          eval_set=[(X_enc_pl[val_orig], y_pl[val_orig])],
          verbose=False)

    if len(val_orig) > 0:
        oof_xgb_pl[val_orig] = m.predict_proba(X_enc_pl[val_orig])
        fold_f1 = macro_f1(y_pl[val_orig], np.argmax(oof_xgb_pl[val_orig], axis=1))
        print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")
    test_xgb_pl += m.predict_proba(X_test_enc) / N_FOLDS

xgb_pl_score = macro_f1(y, np.argmax(oof_xgb_pl, axis=1))
print(f"\n[XGB] OOF F1 (pseudo-labeled): {xgb_pl_score:.4f}")
save_oof("xgb_pl", oof_xgb_pl, test_xgb_pl)

# -----------------------------------------------------------------------
# LightGBM — retrain with pseudo-labeled data
# -----------------------------------------------------------------------
print("\nRetraining LightGBM on pseudo-labeled data...")
oof_lgb_pl  = np.zeros((orig_len, 3))
test_lgb_pl = np.zeros((len(X_test_enc), 3))

for fold, (tr_idx, val_idx) in enumerate(skf_pl.split(X_enc_pl, y_pl)):
    val_orig   = get_orig_val_idx(val_idx, orig_len)
    X_tr, y_tr = sm.fit_resample(X_enc_pl[tr_idx], y_pl[tr_idx])

    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_enc_pl[val_orig], y_pl[val_orig])],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(-1)])

    if len(val_orig) > 0:
        oof_lgb_pl[val_orig] = m.predict_proba(X_enc_pl[val_orig])
        fold_f1 = macro_f1(y_pl[val_orig], np.argmax(oof_lgb_pl[val_orig], axis=1))
        print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")
    test_lgb_pl += m.predict_proba(X_test_enc) / N_FOLDS

lgb_pl_score = macro_f1(y, np.argmax(oof_lgb_pl, axis=1))
print(f"\n[LGB] OOF F1 (pseudo-labeled): {lgb_pl_score:.4f}")
save_oof("lgb_pl", oof_lgb_pl, test_lgb_pl)

# -----------------------------------------------------------------------
# CatBoost — retrain with pseudo-labeled data
# -----------------------------------------------------------------------
print("\nRetraining CatBoost on pseudo-labeled data...")
oof_cat_pl  = np.zeros((orig_len, 3))
test_cat_pl = np.zeros((len(X_test_enc), 3))

for fold, (tr_idx, val_idx) in enumerate(skf_pl.split(X_enc_pl, y_pl)):
    val_orig   = get_orig_val_idx(val_idx, orig_len)
    X_tr, y_tr = sm.fit_resample(X_enc_pl[tr_idx], y_pl[tr_idx])

    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr,
          eval_set=(X_enc_pl[val_orig], y_pl[val_orig]),
          early_stopping_rounds=50)

    if len(val_orig) > 0:
        oof_cat_pl[val_orig] = m.predict_proba(X_enc_pl[val_orig])
        fold_f1 = macro_f1(y_pl[val_orig], np.argmax(oof_cat_pl[val_orig], axis=1))
        print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")
    test_cat_pl += m.predict_proba(X_test_enc) / N_FOLDS

cat_pl_score = macro_f1(y, np.argmax(oof_cat_pl, axis=1))
print(f"\n[CAT] OOF F1 (pseudo-labeled): {cat_pl_score:.4f}")
save_oof("cat_pl", oof_cat_pl, test_cat_pl)

# -----------------------------------------------------------------------
# MLP — retrain with pseudo-labeled data
# -----------------------------------------------------------------------
print("\nRetraining MLP on pseudo-labeled data...")
oof_mlp_pl  = np.zeros((orig_len, 3))
test_mlp_pl = np.zeros((len(X_test_pl_scaled), 3))

for fold, (tr_idx, val_idx) in enumerate(skf_pl.split(X_enc_pl_scaled, y_pl)):
    val_orig   = get_orig_val_idx(val_idx, orig_len)
    X_tr, y_tr = sm.fit_resample(X_enc_pl_scaled[tr_idx], y_pl[tr_idx])

    m = MLPClassifier(**mlp_params)
    m.fit(X_tr, y_tr)

    if len(val_orig) > 0:
        oof_mlp_pl[val_orig] = m.predict_proba(X_enc_pl_scaled[val_orig])
        fold_f1 = macro_f1(y_pl[val_orig], np.argmax(oof_mlp_pl[val_orig], axis=1))
        print(f"  Fold {fold+1:02d} F1: {fold_f1:.4f}")
    test_mlp_pl += m.predict_proba(X_test_pl_scaled) / N_FOLDS

mlp_pl_score = macro_f1(y, np.argmax(oof_mlp_pl, axis=1))
print(f"\n[MLP] OOF F1 (pseudo-labeled): {mlp_pl_score:.4f}")
save_oof("mlp_pl", oof_mlp_pl, test_mlp_pl)

# -----------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------
print("\n" + "=" * 55)
print("PSEUDO-LABELING IMPACT")
print("=" * 55)
print(f"  {'Model':<12} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*40}")
print(f"  {'XGBoost':<12} {xgb_score:>8.4f} {xgb_pl_score:>8.4f} "
      f"{xgb_pl_score-xgb_score:>+8.4f}")
print(f"  {'LightGBM':<12} {lgb_score:>8.4f} {lgb_pl_score:>8.4f} "
      f"{lgb_pl_score-lgb_score:>+8.4f}")
print(f"  {'CatBoost':<12} {cat_score:>8.4f} {cat_pl_score:>8.4f} "
      f"{cat_pl_score-cat_score:>+8.4f}")
print(f"  {'MLP':<12} {mlp_score:>8.4f} {mlp_pl_score:>8.4f} "
      f"{mlp_pl_score-mlp_score:>+8.4f}")

Retraining XGBoost on pseudo-labeled data...
  Fold 01 F1: 0.7793
  Fold 02 F1: 0.7874
  Fold 03 F1: 0.8462
  Fold 04 F1: 0.8027
  Fold 05 F1: 0.7856
  Fold 06 F1: 0.7868
  Fold 07 F1: 0.7880
  Fold 08 F1: 0.7934
  Fold 09 F1: 0.8182
  Fold 10 F1: 0.7864

[XGB] OOF F1 (pseudo-labeled): 0.7984
  Saved: oof_xgb_pl.npy + test_xgb_pl.npy

Retraining LightGBM on pseudo-labeled data...
  Fold 01 F1: 0.7746
  Fold 02 F1: 0.7751
  Fold 03 F1: 0.8473
  Fold 04 F1: 0.7897
  Fold 05 F1: 0.7939
  Fold 06 F1: 0.7710
  Fold 07 F1: 0.7826
  Fold 08 F1: 0.7958
  Fold 09 F1: 0.8306
  Fold 10 F1: 0.7854

[LGB] OOF F1 (pseudo-labeled): 0.7957
  Saved: oof_lgb_pl.npy + test_lgb_pl.npy

Retraining CatBoost on pseudo-labeled data...
  Fold 01 F1: 0.8080
  Fold 02 F1: 0.7889
  Fold 03 F1: 0.8414
  Fold 04 F1: 0.7751
  Fold 05 F1: 0.7444
  Fold 06 F1: 0.7953
  Fold 07 F1: 0.7996
  Fold 08 F1: 0.7916
  Fold 09 F1: 0.7985
  Fold 10 F1: 0.7849

[CAT] OOF F1 (pseudo-labeled): 0.7932
  Saved: oof_cat_pl.npy + test

## Final Ensemble, Thresholds & Submission

In [65]:
# =============================================================================
# FINAL ENSEMBLE + THRESHOLD OPTIMIZATION + SUBMISSION
#
# Ensemble strategy:
#   Weighted average where each model's weight is proportional to its
#   OOF F1 score. Experimentation confirmed this outperforms simple
#   average, rank averaging, power weighting, and calibrated variants.
#
# Threshold optimization:
#   Per-class thresholds are optimized on OOF predictions to maximize
#   macro F1. Thresholds close to 1.0 indicate well-calibrated models.
# =============================================================================

# Load pseudo-labeled OOF predictions
oof_xgb_pl, test_xgb_pl = load_oof("xgb_pl")
oof_lgb_pl, test_lgb_pl = load_oof("lgb_pl")
oof_cat_pl, test_cat_pl = load_oof("cat_pl")
oof_mlp_pl, test_mlp_pl = load_oof("mlp_pl")

xgb_pl_score = macro_f1(y, np.argmax(oof_xgb_pl, axis=1))
lgb_pl_score = macro_f1(y, np.argmax(oof_lgb_pl, axis=1))
cat_pl_score = macro_f1(y, np.argmax(oof_cat_pl, axis=1))
mlp_pl_score = macro_f1(y, np.argmax(oof_mlp_pl, axis=1))

# Weighted ensemble — weights proportional to OOF F1
scores_pl  = np.array([xgb_pl_score, lgb_pl_score, cat_pl_score, mlp_pl_score])
weights_pl = scores_pl / scores_pl.sum()

print("Final ensemble weights:")
print(f"  XGBoost  : {weights_pl[0]:.3f}  (OOF F1: {xgb_pl_score:.4f})")
print(f"  LightGBM : {weights_pl[1]:.3f}  (OOF F1: {lgb_pl_score:.4f})")
print(f"  CatBoost : {weights_pl[2]:.3f}  (OOF F1: {cat_pl_score:.4f})")
print(f"  MLP      : {weights_pl[3]:.3f}  (OOF F1: {mlp_pl_score:.4f})")

oof_ensemble  = (weights_pl[0]*oof_xgb_pl + weights_pl[1]*oof_lgb_pl +
                 weights_pl[2]*oof_cat_pl  + weights_pl[3]*oof_mlp_pl)
test_ensemble = (weights_pl[0]*test_xgb_pl + weights_pl[1]*test_lgb_pl +
                 weights_pl[2]*test_cat_pl  + weights_pl[3]*test_mlp_pl)

ensemble_score = macro_f1(y, np.argmax(oof_ensemble, axis=1))
print(f"\nEnsemble OOF F1 (default)  : {ensemble_score:.4f}")

# Optimize per-class thresholds on OOF predictions
best_thresholds, optimized_f1 = optimize_thresholds(oof_ensemble, y)
print(f"Ensemble OOF F1 (optimized): {optimized_f1:.4f}")
print(f"Thresholds — High: {best_thresholds[0]:.4f}, "
      f"Low: {best_thresholds[1]:.4f}, Medium: {best_thresholds[2]:.4f}")

print("\nFinal per-class breakdown:")
print(classification_report(
    y, apply_thresholds(oof_ensemble, best_thresholds),
    target_names=le_target.classes_
))

# Generate and save final submission
print("Generating submission...")
make_submission(test_ensemble, best_thresholds, "submission_final.csv")

# Log experiment results
log_experiment(
    name="final_4model_pseudo_target_enc",
    model_scores={
        "xgb": xgb_pl_score, "lgb": lgb_pl_score,
        "cat": cat_pl_score,  "mlp": mlp_pl_score
    },
    ensemble_f1=ensemble_score,
    optimized_f1=optimized_f1,
    lb_f1=0.8919,
    notes=(
        "Best submission — 10-fold CV, SMOTE minority, target encoding for country, "
        "pseudo-labeling at 0.95 threshold, 4-model weighted ensemble "
        "(XGB + LGB + CAT + MLP), threshold optimization"
    )
)

# Final summary
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"  XGBoost   OOF F1 : {xgb_pl_score:.4f}")
print(f"  LightGBM  OOF F1 : {lgb_pl_score:.4f}")
print(f"  CatBoost  OOF F1 : {cat_pl_score:.4f}")
print(f"  MLP       OOF F1 : {mlp_pl_score:.4f}")
print(f"  Ensemble  OOF F1 : {ensemble_score:.4f}")
print(f"  Optimized OOF F1 : {optimized_f1:.4f}")
print(f"  Best LB score    : 0.8919")
print("=" * 60)
print(f"\nSubmission saved to: {OUTPUT_DIR / 'submission_final.csv'}")

Final ensemble weights:
  XGBoost  : 0.252  (OOF F1: 0.7984)
  LightGBM : 0.251  (OOF F1: 0.7957)
  CatBoost : 0.251  (OOF F1: 0.7932)
  MLP      : 0.246  (OOF F1: 0.7771)

Ensemble OOF F1 (default)  : 0.8012
Ensemble OOF F1 (optimized): 0.8025
Thresholds — High: 0.9855, Low: 1.0625, Medium: 0.9912

Final per-class breakdown:
              precision    recall  f1-score   support

        High       0.82      0.62      0.71       470
         Low       0.91      0.93      0.92      6280
      Medium       0.79      0.78      0.78      2868

    accuracy                           0.87      9618
   macro avg       0.84      0.78      0.80      9618
weighted avg       0.87      0.87      0.87      9618

Generating submission...
  Saved: outputs\submission_final.csv
  Prediction distribution:
Low       1601
Medium     715
High        89
  Experiment logged → outputs\experiment_log.json

FINAL SUMMARY
  XGBoost   OOF F1 : 0.7984
  LightGBM  OOF F1 : 0.7957
  CatBoost  OOF F1 : 0.7932
  MLP  